### Auto-Generated Code Cells

The following code cells contain code from external .py files that has been automatically added to remove these dependencies. This allows you to run this version of the SELENE notebook on Cloud platforms such as Google Colab. Run all following code cells before starting with the actual notebook content. 

Most Cloud platforms such as Google Colab have a wide range of Python packages preinstalled. However, if any import statements in the following code cells throws an error because of a missing package, you can create and run a new code cell to install the missing package using the `pip` command as shown here:
```
!pip install <package-name>
```

You can use the code cell below for convenience.

In [ ]:
#!pip install <package-name>

In [ ]:
# === Inline content for module src.utils.compute.gpu  ==

# === Inlined imports ===
import random
import torch

# === Embedded configs ===


# === Inlined definitions ===
def select_device(force_cpu=False, required_memory_gb=None):
    if force_cpu == True:
        return torch.device('cpu')
    candidate_gpus = []
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(device=i)
        if required_memory_gb is None or free / (1024 * 1024 * 1024) > required_memory_gb:
            candidate_gpus.append((i, free))
    if len(candidate_gpus) == 0:
        return torch.device('cpu')
    else:
        device = torch.cuda.device(random.choice(candidate_gpus)[0])
        return torch.device(f'cuda:{device.idx}')

In [ ]:
# === Inline content for module src.utils.data.files  ==

# === Inlined imports ===
import requests
import bz2
from tqdm import tqdm
import os, re, yaml
import zipfile, tarfile

# === Embedded configs ===
CONFIG = yaml.safe_load("""urls:
  downloads:
    notebooks: "https://github.com/chrisvdweth/selene/tree/master/notebooks"
    datasets: "https://chrisvdw.net/projects/selene/downloads/datasets/"
    models: "https://chrisvdw.net/projects/selene/downloads/models/"
    """)

# === Inlined definitions ===
def create_folder(folder_name, exist_ok=True):
    try:
        os.makedirs(folder_name, exist_ok=exist_ok)
        return folder_name
    except:
        return None

# Fine-Tuning BERT for Named Entity Recognition

This notebook shows how to fine-tune a pretrained BERT model for Named Entity Recognition (NER), which serves as a representative example of a sequence labeling task, where every token in a text receives its own label. The goal is not to reach state-of-the-art results but to understand all the steps that fine-tuning requires, from the data to the final predictions. To keep everything fast and easy to follow, we use CoNLL-2003, a small but popular English NER benchmark that fits comfortably into memory, together with the compact bert-base-cased model.

Along the way, we walk through the full workflow: recapping the BERT architecture and the NER task, preparing the data (mapping labels to indices, tokenizing into subwords, and aligning word-level labels with subword tokens), batching with dynamic padding, and evaluating with entity-level metrics. We then fine-tune the model in two ways. The “manual” approach, with our own optimizer, scheduler, and training loop, exposes each individual step, while the Trainer class shows the more practical route that achieves the same result with far less code. Finally, we use the fine-tuned models to extract entities from new text, first with our own post-processing and then with the transformers pipeline.

### Setting up the Notebook

#### Make Required Imports

This notebook requires the import of different Python packages but also additional Python modules that are part of the repository. If a package is missing, use your preferred package manager (e.g., [conda](https://anaconda.org/anaconda/conda) or [pip](https://pypi.org/project/pip/)) to install it. If the code cell below runs with any errors, all required packages and modules have successfully been imported.

In [1]:
import re
import gc
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline,
    get_linear_schedule_with_warmup
)

import torch
from torch.utils.data import DataLoader

from sklearn.metrics import precision_score, recall_score, f1_score
from seqeval.metrics import classification_report, f1_score, recall_score, precision_score


#### Create Output Folder

When fine-tuning a BERT model, we periodically save the current state of the model to disk. Although the dataset and model used in this notebook is relatively small, we still adopt this best practice of saving the model to avoid losing the training progress in case of any errors or crashes. You can change the output folder by editing the code cell below.

In [2]:
output_folder = create_folder("data/generated/models/bert/")

print(f"Output folder: {output_folder}")

Output folder: data/generated/models/bert/


#### Checking & Setting Computing Device

PyTorch allows to train neural networks on supported GPU to significantly speed up the training process. If you have a support GPU, feel free to utilize it. However, for this notebook it's certainly not needed as our dataset is small and our network model is very simple. We provide an auxiliary method to automatically select the best device. It checks if a supported GPU is available and if so, uses it as the preferred device.

In [3]:
# Select preferred device (GPU, if available; CPU otherwise); you can enfore the use of the CPU
DEVICE = select_device(force_cpu=False)

print("Available device: {}".format(DEVICE))

Available device: cuda:0


#### Preliminaries

This notebook assume a basic understanding of the BERT model and its pretraining.

---

## Introduction

### Named Entity Recognition (NER)

**Named Entity Recognition (NER)** is the task of finding and classifying spans of text that refer to real-world entities, such as persons, organizations, locations, dates, or domain-specific types like genes or drugs. Given a sentence like *"John lives in Singapore"*, an NER system should identify *"John"* as a person and *"Singapore"* as a location. It is a core information extraction task, used for example in search, question answering, and knowledge graph construction. Since entities can span several words, NER is usually framed as a **token classification** problem: each token receives a label, commonly in the BIO scheme, where `B-` marks the beginning of an entity, `I-` a token inside it, and `O` a token outside any entity. The table below shows a slightly complex example.

| Index | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| **Token** | John | Smith | lives | Singapore | and | works | at | the | National | University | of | Singapore | since | 2020 | . |
| **Label** | B-PER | I-PER | O | B-LOC | O | O | O | O | B-ORG | I-ORG | I-ORG | I-ORG | O | O | O |

**Important** NER has no single universal annotation scheme: which entity types exist and how spans are encoded depends on the dataset and application. Some corpora use coarse types like persons, organizations, and locations, others add dates, quantities, or products (e.g., OntoNotes with 18 types), and domain-specific datasets define types such as genes, diseases, or legal references. The span encoding also varies, with BIO (also called IOB2) being common, alternatives like BIOES adding explicit end and single-token tags, and the label set changes the size of the classification head accordingly. In this notebook we assume the label set defined by our training set, the CoNLL-2003 style scheme: `['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']`, i.e., nine labels in total.

Identifying and extracting named entities is a fundamental NLP task that is used for a wide range of applications. The table below provides an overview of common applications and how they used named entities as part of their workflow.

| Application | How named entities are used |
|---|---|
| **Information extraction & knowledge graphs** | Entities become nodes in a knowledge graph; together with relation extraction they populate structured databases from unstructured text. |
| **Search engines & semantic search** | Recognizing entities in queries and documents improves retrieval, entity-based filtering, and result snippets (e.g., "Apple" the company vs. the fruit). |
| **Question answering & chatbots** | Entities in a question (people, places, dates) identify what is asked about and fill slots in dialogue systems, such as a destination or date in a booking assistant. |
| **Document classification & content recommendation** | Entities serve as features for topic tagging, and news or articles can be linked and recommended based on shared entities. |
| **News monitoring & media analytics** | Tracking mentions of companies, politicians, or events over time supports trend detection and sentiment analysis per entity. |
| **Finance & compliance** | Extracting company names, monetary amounts, and counterparties from reports and filings supports risk analysis, KYC/AML checks, and sanctions screening. |
| **Healthcare & biomedical text mining** | Identifying diseases, drugs, genes, and dosages in clinical notes and literature supports cohort selection, pharmacovigilance, and literature-based discovery. |
| **Legal & contract analysis** | Extracting parties, dates, jurisdictions, and statute references speeds up contract review and e-discovery. |
| **Privacy & data anonymization** | Detecting personal names, addresses, and IDs enables automatic redaction or pseudonymization of sensitive documents. |
| **Recruiting & resume parsing** | Extracting names, employers, schools, skills, and locations structures resumes for matching candidates to jobs. |
| **Customer support & CRM** | Pulling product names, order numbers, and company names from emails and tickets supports routing and automatic ticket enrichment. |
| **Machine translation & text-to-speech** | Recognizing names helps decide what should be transliterated or left untranslated and how to pronounce it. |

NER has been approached with a wide range of machine learning methods over the years. Early statistical systems relied on sequence models such as Hidden Markov Models and Conditional Random Fields, which use hand-crafted features and model label dependencies between neighboring tokens. These were followed by RNN-based neural networks, typically BiLSTM-CRF architectures, that learn features automatically from word and character embeddings. Modern approaches, however, are built on the Transformer architecture: pretrained encoders such as BERT produce rich contextual representations for every token, and fine-tuning them, which is the approach we follow in this notebook.

### BERT in a Nutshell

To understand how fine-tuning a pretrained BERT model works and what it requires from our data, it helps to first recap BERT's basic idea and architecture. Fine-tuning reuses the pretrained encoder and only adds a small task-specific head, so knowing what the encoder expects as input (tokenized, single or paired segments with special tokens) and what it produces (a contextual embedding for every token) explains why the dataset must be prepared in a particular way and why a token classification head can sit directly on top of its outputs.

#### Goal: Contextual Word Embeddings

BERT's goal is to learn **contextual word embeddings**: instead of assigning each word one fixed vector, it computes a representation for every token that depends on the entire sentence. The word *"bank"* therefore gets different vectors in *"river bank"* and *"bank account"*, which lets the model handle ambiguity, syntax, and long-range dependencies far better than static embeddings such as Word2Vec. These embeddings are so useful because they are learned from huge amounts of unlabeled text through the proxy tasks MLM and NSP &mdash; see below &mdash; and can then be reused for many tasks, either as features for other models or by fine-tuning the pretrained encoder with a small task-specific head on limited labeled data.

Architecturally, BERT is based on an **encoder-only Transformer**, whose self-attention lets every token attend directly to every other token in the sequence, in both directions. This typically outperforms earlier contextual models such as ELMo, which relies on recurrent networks (LSTMs). Those process text sequentially, are harder to parallelize, and struggle more with long-range dependencies. Moreover, ELMo only concatenates separately trained left-to-right and right-to-left representations, whereas BERT's encoder conditions on both sides jointly in every layer. The result is richer contextual embeddings and, in practice, stronger performance across a wide range of NLP tasks.

#### Learning Objectives

##### Masked Language Model (MLM)

The MLM learning objective directly implements the **Distributional Hypothesis** which says that a word's meaning is determined by the contexts in which it appears. MLM implements this almost literally: a word is hidden, and the model must predict it using only the surrounding words, left and right. To do this well across billions of tokens, the network has to learn representations in which words that occur in similar contexts are treated similarly, since they must lead to similar predictions. More specifically, during training, a fixed proportion of input tokens (typically around 15%) is selected at random and their original identities are hidden from the model. Of those 15% selected tokens:
  * 80% are replaced with a `[MASK]` token,
  * 10% are replaced with a random token,
  * 10% are left unchanged.

Only, those ~15% of selected tokens the model considered when computing the loss during training. To show an example, consider the following simple sentence:

> I watched a movie last night on an old DVD I found about a man who lost his memory.

As output, we might get the following corrupted sequence:

> I watched a <font color="red"><b>[MASK]</b></font> last night on an old <font color="red"><b>river</b></font> I found about a <font color="red"><b>[MASK]</b></font> who lost his <font color="red"><b>memory</b></font>.

Here, a total of four tokens have been selected. Both *"watched"* and *"man"* have been masked (i.e., replaced with the `[MASK]` token). *"DVD"* has been replaced with the random token *"banana"*, and *"memory"* has been kept as is. The `[MASK]` token appears only during pre-training and never in downstream tasks, so if every selected token were masked, the model would only learn good representations at `[MASK]` positions. Replacing some selected tokens with a random word or leaving them unchanged fixes this: since the model cannot tell which visible tokens are targets or corrupted, it must build a context-aware representation for every token and check each one against its context. This reduces the mismatch between pre-training and fine-tuning, where all inputs are real tokens.

##### Next Sentence Prediction (NSP):

A training sample for BERT contains pairs of sentences. In ~50% of the cases, the second sentence truly follows the first one in the original corpus; in the other 50%, it is a randomly sampled sentence from the corpus. The two sentences are concatenated into a single input sequence, separated by a special `[SEP]` token, and a classification token `[CLS]` is added at the beginning. The model is then trained to predict whether the second sentence is the actual continuation of the first. The goal of NSP is to encourage BERT to capture inter-sentential coherence and discourse-level information, which is important for tasks involving sentence pairs, such as question answering and natural language inference. By framing this as a binary classification problem over the `[CLS]` representation, BERT learns to encode not only the meaning of individual sentences but also how sentences relate to each other in context. 

The figure below illustrates the overall training setup of BERT with both learning objectives. This example assumes that this sentence pair does indeed represent two adjacent sentences in the same document, and the binary class label for the NSP is $0$; for unrelated sentences the label is $1$ (i.e., "is random"). Similar to the previous example, a total of four tokens have been selected for the MLM learning objective, two are masked, one is replaced with a random token, and one is left unchanged.

<img src="https://raw.githubusercontent.com/chrisvdweth/selene/refs/heads/master/notebooks/images/illustrations/bert/bert-example-setup.png" style="margin:auto;max-width:900px;width:100%" alt="BERT - Example Setup" />

<br />

Of course, the fine-tuning samples look different from pretraining samples because the objectives differ: pretraining generates its own training signal from raw text, using sentence pairs for NSP and corrupting about 15% of tokens (masking or replacing them) for MLM, whereas NER fine-tuning is a supervised task with human-annotated labels, so each sample contains only a single segment (there is no second segment and no NSP) and the input is left unmasked and unreplaced, since hiding a word would hide the very evidence for its label, with the targets being one label per token (`B-PER`, `O`, ...) rather than the original tokens. The details, such as aligning word-level labels with WordPiece subtokens and ignoring special tokens in the loss, will be explained when we prepare the dataset for fine-tuning.

---

## Data Preparation

### Downloading Corpus

In this notebook, we will be using the CoNLL-2003 dataset, a widely used benchmark dataset for NER. It consists of English news articles from the Reuters corpus, split into roughly 14,000 training, 3,250 validation, and 3,450 test sentences. Each token is annotated with a part-of-speech tag, a syntactic chunk tag, and an NER tag using four entity types: persons (`PER`), organizations (`ORG`), locations (`LOC`), and miscellaneous (`MISC`), encoded in the BIO scheme. Because of its clean annotation and long history of reported results, it is the standard testbed for comparing NER models, and it is readily available through the Hugging Face `datasets` library as `conll2003`. So let's load it.

In [4]:
dataset = load_dataset("tomaarsen/conll2003")

The code cell below extracts all NER labels used in this dataset.

In [5]:
label_list = dataset["train"].features["ner_tags"].feature.names

print(f"List of labels: {label_list}")

List of labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In the dataset, the NER tags are not stored as label strings like `B-PER` but as integer indices into the label list (e.g., `1` for `B-PER`), which is more compact and is the format the model needs for its loss computation. To inspect the data, interpret predictions, and tell the model what each output neuron means, we need to convert easily between the two representations. We can do this with two dictionaries: `label2id`, mapping each label string to its index, and `id2label`, mapping each index back to its label string.

In [6]:
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

print(id2label)
print(label2id)

{0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC', 7: 'B-MISC', 8: 'I-MISC'}
{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}


To show see how an annotated sentence looks like, let's consider the first sentence of the training data and print the token together with its annotation (the numerical label and the string label); we can ignore all the other annotations for each token (e.g., the part-of-speech tags) since we will not use them here.

In [7]:
tokens = dataset["train"][0]["tokens"]
ner_tags = dataset["train"][0]["ner_tags"]

for token, label in zip(tokens, ner_tags):
    print(f"{token:9s} {label}={id2label[label]}")

EU        3=B-ORG
rejects   0=O
German    7=B-MISC
call      0=O
to        0=O
boycott   0=O
British   7=B-MISC
lamb      0=O
.         0=O


CoNLL-2003 is a carefully annotated, widely studied benchmark that is already tokenized at the word level and comes with consistent BIO labels, so no explicit cleaning is required for this notebook. We therefore skip steps like deduplication, label correction, or text normalization and can move directly to preparing the data for the model. (A few annotation inconsistencies are known to exist, as in most benchmarks, but they are rare enough not to matter for our purposes.)

### Tokenize & Align Labels

CoNLL-2003 sentences are tokenized and annotated at the word level, with exactly one label per word. BERT, however, relies on subword (WordPiece) tokenization, so rare or long words are split into several pieces (e.g., "Smithers" might become `Smith`, `##ers`). This means we have to change both the tokens and the labels: first, we split words into subwords wherever the pretrained tokenizer requires it, and second, whenever a word is split, we align the word-level labels with the new subword sequence, since the number of tokens no longer matches the number of labels. The special tokens `[CLS]` and `[SEP]` added by the tokenizer also need labels, and how we handle all of this will be explained when we prepare the dataset.

To this end, we first need to load the matching tokenizer for the model we will be using. `bert-base-cased` is the base-sized BERT model, pretrained on English Wikipedia and BookCorpus, together with its WordPiece tokenizer, which has a vocabulary of about 29,000 subwords. The "cased" variant preserves the original capitalization of the text, whereas the uncased variant lowercases everything (and strips accents) before tokenization. For NER this distinction matters because capitalization is one of the strongest cues for named entities: *"Apple"* at the start of a mid-sentence phrase or *"May"* versus *"may"* often signals a proper noun, and words like *"Paris"*, *"Reuters"*, or *"Smith"* are typically capitalized precisely because they are names.

In [8]:
MODEL_NAME = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

When a word is split into several subwords, the single word-level label no longer maps one-to-one onto the tokens, so we must decide how to distribute it. Here we follow the standard approach: the first subword receives the word's original label, and all subsequent subwords (as well as special tokens like `[CLS]`, `[SEP]`, and padding) receive `-100`, the index that PyTorch's cross-entropy loss ignores by default. For example, if "Smithers" is split into `Smith` and `##ers` and the word is labeled `B-PER`, the aligned sequence looks like this:

| Index | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| **Token** | [CLS] | Smith | ##ers | lives | in | Paris | [SEP] |
| **Label** | -100 | B-PER | -100 | O | O | B-LOC | -100 |

This is not a problem because `-100` only affects the loss, not the forward pass: all subwords, including `##ers`, are still fed to the model and take part in the self-attention computations, so the model uses their information when building the contextual representation of the first subword. Since the prediction for the word is read from its first subword's output, which has already attended to the remaining subwords, no information is lost, and each word contributes exactly one term to the loss.

The method `tokenize_and_align_labels()` implements this idea. It takes a batch of CoNLL-2003 examples and converts them into model-ready inputs in two steps. First, it runs the BERT tokenizer on the pre-split words (`is_split_into_words=True`), which breaks words into subwords where needed, adds `[CLS]` and `[SEP]`, and truncates sequences to 256 tokens. Second, it builds a matching `labels` list for each sentence by using the tokenizer's `word_ids()`, which tells for every subword which original word it came from: special tokens (no word ID) get `-100`, the first subword of each word receives the word's original NER label, and any continuation subwords also get `-100`. The result is that `input_ids`, `attention_mask`, and `labels` all have the same length per sentence, with the labels aligned to the subword tokens as described above.

In [9]:
def tokenize_and_align_labels(examples):
    # Tokenize word into subwords (if required)
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=256,
        is_split_into_words=True,
    )
    # Align labels for word that have been split
    all_labels = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        labels, previous_word_id = [], None
        for word_id in word_ids:
            if word_id is None:  # [CLS], [SEP], padding
                labels.append(-100)
            elif word_id != previous_word_id:  # first sub-token of a word
                labels.append(word_labels[word_id])
            else:  # continuation sub-token
                labels.append(-100)
            previous_word_id = word_id
        all_labels.append(labels)
    tokenized["labels"] = all_labels
    return tokenized

The method is designed to be applied to the whole dataset in batches (e.g., via `dataset.map(..., batched=True)`) as done in the code cell below.

In [10]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

Let's have a look at the same example sentence as before. This sentence contains only a single word that was split into subwords, but we can see that only the first subword gets the original NER label while all subsequent subwords are now annotated with `-100`. Apart from that, the method also added the required `[CLS]` token and the front and the required `[SEP]` token at the end. Notice that the sample actually contains the token IDs and not the tokens (i.e., subwords) themselves. We therefore use the tokenizer to decode the IDs into readable tokens.

In [11]:
token_ids = tokenized_dataset["train"][0]["input_ids"]
label_ids = tokenized_dataset["train"][0]["labels"]

for token_id, label in zip(token_ids, label_ids):
    print(f"{tokenizer.decode(token_id):9s} {label}")

[CLS]     -100
EU        3
rejects   0
German    7
call      0
to        0
boycott   0
British   7
la        0
##mb      -100
.         0
[SEP]     -100


All sentences in the dataset are now in the correct forward to pass to the pretrained BERT model.

We fine-tune the model in batches, meaning several training samples at a time, because this makes much better use of the GPU and gives smoother gradient estimates than updating on one sample at a time. However, a batch is a single tensor, which assumes that all samples have the same length, while our tokenized sentences naturally differ in length. We therefore have to pad the shorter inputs to the length of the longest input in the batch (rather than to a fixed global maximum, which would waste computation). Conveniently, `DataCollatorForTokenClassification` performs this **dynamic padding** for us: it pads `input_ids` with the pad token, sets the `attention_mask` to 0 at padded positions so they are ignored by attention, and pads the `labels` with `-100` so that the padded positions do not contribute to the loss.

In [12]:
data_collator = DataCollatorForTokenClassification(tokenizer)

With the data collator in place, we can now create `DataLoader` instances for the training and validation data. The `DataLoader` takes care of the practical side of feeding data to the model: it groups samples into batches, shuffles the training data at every epoch so the model does not see the examples in a fixed order, and applies the data collator passed as `collate_fn` to each batch, which performs the dynamic padding of `input_ids`, `attention_mask`, and `labels`. The validation loader typically does not need shuffling, since the order does not affect the evaluation results, and it can often use a larger batch size because no gradients need to be stored.

In [13]:
BATCH_SIZE = 32

train_loader = DataLoader(tokenized_dataset["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator)
val_loader   = DataLoader(tokenized_dataset["validation"], batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_loader  = DataLoader(tokenized_dataset["test"], batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

**Side note:** CoNLL-2003 is small enough to fit comfortably into RAM: with roughly 14,000 training sentences of about 14 words each, the whole dataset amounts to only a few megabytes, even after tokenization. This greatly simplifies data management, since we can load and preprocess everything up front (e.g., with a single `dataset.map(...)` call) and keep the tokenized samples in memory. We therefore do not need streaming, sharded files, or on-the-fly loading from disk, and the `DataLoader` can simply draw batches from the in-memory dataset. For much larger corpora, such as those used for pretraining, this would not be possible, and more elaborate data pipelines would be required.

---

## Model Fine-Tuning

We fine-tune a pretrained BERT model using the `AutoModelForTokenClassification` class from the `transformers` library, which loads the pretrained `bert-base-cased` encoder and attaches a new, randomly initialized token classification head with `num_labels=9` outputs (together with our `label2id` and `id2label` mappings). For the actual training, we consider two approaches. First, in the **"manual" approach**, we define the optimizer and the learning rate scheduler ourselves and implement the training loop explicitly (forward pass, loss computation, backward pass, parameter update, and evaluation), which lets us focus on the individual steps that fine-tuning requires. Second, we use the **`Trainer` class**, which builds on the utility classes of the `transformers` library and handles batching, optimization, scheduling, evaluation, and checkpointing for us. This is the more practical method, as it greatly reduces the amount of code we have to write while achieving the same result.

### Approach 1: "Manual" Training

#### Loading the Pretrained Model

We first need to load the pretrained BERT model that we fine-tune for NER. In the code cell below `AutoModelForTokenClassification.from_pretrained` loads the pretrained encoder weights for `MODEL_NAME` (here `bert-base-cased`) and attaches a new, randomly initialized token classification head, so the library will warn that the classifier weights are newly initialized, which is expected since they are learned during fine-tuning. 

The argument `num_labels=len(label_list)` sets the size of this head to one output per label (nine for CoNLL-2003), while `id2label` and `label2id` store our label mappings in the model configuration, so that predictions can later be interpreted as label strings such as `B-PER` instead of bare indices, and the mappings are saved along with the model. Finally, `.to(DEVICE)` moves all model parameters to the chosen device (e.g., a GPU), which is required so that the model and the input batches reside on the same device and training runs efficiently.

In [14]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

**Side note:** By default, we fine-tune the model by updating all of its parameters during training, which is the common approach for NER because the encoder needs to adapt to token-level, often domain-specific features. An alternative is to train only the classification head and freeze the embedding and encoder layers, which is faster and needs less memory but usually costs some accuracy. This can be done with a simple code snippet that disables gradient computation for the base model:

In [15]:
#for param in model.base_model.parameters():
#    param.requires_grad = False

Again, by default, we do not use this snippet in our experiments, so the whole model is fine-tuned. However, you are welcome to freeze the base model and see how it affects the results.

#### Create Optimizer & Scheduler

For the manual training loop we need an optimizer and a learning rate scheduler, and we wrap their setup in two small helper methods to keep the training code clean and reusable. `make_optimizer` creates an AdamW optimizer, the standard choice for fine-tuning Transformers, and follows the original BERT recipe by applying weight decay only to the weight matrices, while excluding biases and LayerNorm weights, which are not meaningfully regularized by shrinking them toward zero. `linear_warmup_decay` creates the learning rate scheduler: the learning rate increases linearly from zero during a short warmup phase (a fraction of the total steps) and then decays linearly back to zero. The warmup avoids large, destabilizing updates at the start while the new classification head is still random, and the decay allows the model to settle into a good solution towards the end of training.

In [16]:
def make_optimizer(model, lr, weight_decay):
    """Create an AdamW optimizer, excluding biases and LayerNorm weights from weight decay."""
    no_decay = ["bias", "LayerNorm.weight"]
    param_groups = [
        {
            "params": [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
        },
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]
    return torch.optim.AdamW(param_groups, lr=lr)

def linear_warmup_decay(optimizer, total_steps, warmup_ratio):
    """Create a linear warmup and decay learning rate scheduler."""
    warmup_steps = int(warmup_ratio * total_steps)
    return get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

Let's define the hyperparameters that both the "manual" training loop and the `Trainer` approach need, so we set them once and keep the two comparable. `EPOCHS` is the number of passes over the training data (here just one, which keeps the run short, and is typically increased to 3-5 for the best results). `LR` is the peak learning rate of 3e-5, a small value in the usual range for fine-tuning BERT, so that the pretrained weights adapt without being destroyed. `WEIGHT_DECAY` of 0.01 sets the strength of the regularization applied to the weight matrices, and `WARMUP_RATIO` of 0.1 means the learning rate rises linearly during the first 10% of the training steps before decaying linearly to zero.

In [17]:
EPOCHS = 5
LR = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

With that, we can now create the optimizer and scheduler instances.

In [18]:
total_steps = EPOCHS * len(train_loader)
warmup_steps = int(WARMUP_RATIO * total_steps)

optimizer = make_optimizer(model, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = linear_warmup_decay(optimizer, total_steps=total_steps, warmup_ratio=WARMUP_RATIO)

#### Fine-Tuning

During training, we periodically assess progress by evaluating the current model on the validation data, which are not used for updating the weights. This shows whether the model is actually learning to generalize and not just fitting the training set, and it lets us pick good hyperparameters or checkpoints. The `evaluate()` function does this in a few steps. It switches the model to evaluation mode (`model.eval()`, which disables dropout) and loops over the validation loader, moving each batch to the device and running a forward pass. Because the batch contains labels, the model also returns the loss, which we accumulate to report the average validation loss. 

The predicted label for each token is the one with the highest logit (`argmax`). We then drop all positions labeled `-100` (special tokens, padding, and continuation subwords), so that only one prediction per word remains, and convert both the predicted and the true label IDs into tag strings such as `B-PER`. Finally, the function computes entity-level precision, recall, and F1 with `seqeval`, and returns them together with the loss and the raw predictions and references, which can be used for further analysis such as a per-entity-type report.

In [19]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_refs, total_loss, n_batches = [], [], 0.0, 0

    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        total_loss += out.loss.item()
        n_batches += 1

        preds = out.logits.argmax(dim=-1).cpu().numpy()     # (batch, seq_len)
        labels = batch["labels"].cpu().numpy()              # (batch, seq_len)

        # Keep only positions with a real label (drop -100), converting IDs to tag strings
        for p_seq, l_seq in zip(preds, labels):
            all_preds.append([label_list[p] for p, l in zip(p_seq, l_seq) if l != -100])
            all_refs.append([label_list[l] for p, l in zip(p_seq, l_seq) if l != -100])

    return {
        "loss": total_loss / n_batches,
        "precision": precision_score(all_refs, all_preds),
        "recall": recall_score(all_refs, all_preds),
        "f1": f1_score(all_refs, all_preds),
        "preds": all_preds,
        "refs": all_refs,
    }

**Important:** Evaluating predicted NER labels is not trivial. For example, consider the following ground-truth label `B-ORG I-ORG I-ORG I-ORG` (National University of Singapore). It is arguably not obvious how we should treat a prediction such as, say, `B-ORG I-ORG I-ORG I-PER`. While it is clearly not (fully) correct, it is also not completely wrong, and in fact, there are more or less strict approaches to evaluate the different metrics. Our example implementation here is all or nothing at the entity level. `seqeval` follows the CoNLL evaluation convention (exact match): a predicted entity counts as a true positive only if both its **span boundaries** and its **type** exactly match a gold entity. There is no partial credit for getting some of the tokens right. In more detail, this it how it works:

* The tag sequence is first converted into entity spans, e.g. `B-ORG I-ORG I-ORG I-ORG` becomes `(ORG, start=8, end=11)`.
* The set of predicted spans is compared with the set of gold spans.
* A span in both sets is a true positive (TP); a predicted span not in the gold set is a false positive (FP); a gold span not in the predicted set is a false negative (FN).

The table below outlines a few example predictions for our example and provides their different interpretations.

| Prediction | Result |
|---|---|
| `B-ORG I-ORG I-ORG I-ORG` | 1 TP |
| `B-ORG I-ORG I-ORG O` (last word missed) | 1 FP and 1 FN, since the span differs |
| `B-ORG I-ORG I-ORG I-LOC` | the ORG span (3 words) is wrong, and a spurious LOC span is predicted: 2 FP and 1 FN |
| `B-LOC I-LOC I-LOC I-LOC` | wrong type: 1 FP and 1 FN |

In other words, a partially correct entity is penalized twice: it counts as a missed entity (hurting recall) and as a wrong prediction (hurting precision). Subword splits are not a separate concern in our setup, because `evaluate()` keeps only the first subword of each word, so `seqeval` sees one tag per word and judges entities at the word level. Only `overall_accuracy` is token-level and gives partial credit, whereas precision, recall, and F1 are strict entity-level metrics, which is why they are usually lower than accuracy and more informative for NER. Finally, by default `seqeval` handles malformed sequences leniently (an `I-X` without a preceding `B-X` starts a new entity), while `mode="strict"` with `scheme="IOB2"` rejects them, and `seqeval` offers no partial-match variant like some other evaluation schemes (e.g., SemEval-2013). However, a deeper discussion about different evaluation schemes for NER labels is beyond our scope here.

We can now perform the actual fine-tuning. This code below implements the basic training loop that fine-tunes the model by repeatedly showing it batches and nudging the weights to reduce the loss. For each epoch, the model is put into training mode (`model.train()`, which enables dropout), and for each batch from the `train_loader` we perform the standard steps: move the batch to the device, reset the gradients (`zero_grad()`), run the forward pass (which returns the loss because the batch contains labels), compute the gradients with `loss.backward()`, clip their norm to 1.0 to avoid unstable updates, and update the weights with `optimizer.step()`. The `scheduler.step()` then advances the learning rate according to the warmup and decay schedule.

Along the way, the loss is recorded in `history` and a running average is printed every `LOG_EVERY` step, so we can monitor whether training is progressing. At the end of each epoch, we call `evaluate()` on the validation data to track validation loss, precision, recall, and F1, which shows how well the model generalizes. Finally, the current model weights are saved to disk, so the fine-tuned model can be reloaded later without retraining.

In [20]:
history = {"step_loss": [], "val_loss": [], "val_f1": []}
LOG_EVERY = 50

for epoch in range(EPOCHS):
    model.train()
    running = 0.0

    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        optimizer.zero_grad()
        out = model(**batch)                                      
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) 
        optimizer.step()
        scheduler.step()

        history["step_loss"].append(loss.item())
        running += loss.item()
        if step % LOG_EVERY == 0:
            print(f"Epoch {epoch+1}, Step {step}/{len(train_loader)}  loss: {running/LOG_EVERY:.4f}")
            running = 0.0

    metrics = evaluate(model, val_loader)
    history["val_loss"].append(metrics["loss"])
    history["val_f1"].append(metrics["f1"])
    print(f"==> Epoch {epoch+1}: val loss {metrics['loss']:.4f} | "
          f"P {metrics['precision']:.4f} R {metrics['recall']:.4f} F1 {metrics['f1']:.4f}\n")

    # Save current model
    torch.save(model.state_dict(), f"{output_folder}bert_ner.pt")

Epoch 1, Step 50/439  loss: 1.8732
Epoch 1, Step 100/439  loss: 0.5122
Epoch 1, Step 150/439  loss: 0.1815
Epoch 1, Step 200/439  loss: 0.1069
Epoch 1, Step 250/439  loss: 0.0772
Epoch 1, Step 300/439  loss: 0.0636
Epoch 1, Step 350/439  loss: 0.0651
Epoch 1, Step 400/439  loss: 0.0543
==> Epoch 1: val loss 0.0439 | P 0.9202 R 0.9251 F1 0.9226

Epoch 2, Step 50/439  loss: 0.0333
Epoch 2, Step 100/439  loss: 0.0333
Epoch 2, Step 150/439  loss: 0.0362
Epoch 2, Step 200/439  loss: 0.0274
Epoch 2, Step 250/439  loss: 0.0285
Epoch 2, Step 300/439  loss: 0.0330
Epoch 2, Step 350/439  loss: 0.0285
Epoch 2, Step 400/439  loss: 0.0287
==> Epoch 2: val loss 0.0349 | P 0.9436 R 0.9519 F1 0.9477

Epoch 3, Step 50/439  loss: 0.0142
Epoch 3, Step 100/439  loss: 0.0175
Epoch 3, Step 150/439  loss: 0.0171
Epoch 3, Step 200/439  loss: 0.0170
Epoch 3, Step 250/439  loss: 0.0146
Epoch 3, Step 300/439  loss: 0.0150
Epoch 3, Step 350/439  loss: 0.0137
Epoch 3, Step 400/439  loss: 0.0152
==> Epoch 3: val lo

Making predictions with the fine-tuned model is more involved than a single forward pass because the model outputs one label per subword token, whereas what we actually want are complete named entities, which may span several words and several subwords. The method therefore has to map subword predictions back to words and merge consecutive BIO tags into entity spans with character positions in the original text. It works in six steps:

1. **Word splitting:** a regular expression splits the text into words and punctuation marks (e.g., "Paris." becomes "Paris" and "."), mimicking the pre-split format of CoNLL-2003, and remembers each word's character offsets.
2. **Subword tokenization:** the words are passed to the tokenizer with `is_split_into_words=True`, and `word_ids` records which word each subword belongs to (`None` for `[CLS]` and `[SEP]`).
3. **Forward pass:** the model produces logits for every token, which are converted into probabilities with softmax, and we keep the most likely label and its probability (the confidence) per token.
4. **One prediction per word:** as in training, only the first subword of each word is used, so every word gets exactly one tag and one score. Words cut off by truncation default to `O`.
5. **Merging BIO tags:** the method walks through the words and starts a new entity at each `B-` tag (or at an `I-` tag that does not continue an entity of the same type), extends the current entity on matching `I-` tags, and closes it when it meets an `O`. The character start and end positions are updated along the way.
6. **Output formatting:** each entity is returned with its text sliced from the original string, its label, its character offsets, and the mean confidence of its words.

The `@torch.no_grad()` decorator disables gradient tracking, since we only run inference, and `model.eval()` turns off dropout.

In [21]:
@torch.no_grad()
def predict_entities(text, model=model, tokenizer=tokenizer, max_length=512):
    model.eval()
    id2label = model.config.id2label

    # 1. Split into words and punctuation, remembering character offsets.
    #    NER datasets like CoNLL are pre-split this way ("Paris." -> "Paris", "."), so we match that.
    matches = list(re.finditer(r"\w+|[^\w\s]", text))
    words = [m.group() for m in matches]
    if not words:
        return []

    # 2. Tokenize the words into sub-word tokens
    enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=max_length, return_tensors="pt")
    word_ids = enc.word_ids(0)          # maps each sub-token to its word index (None for [CLS]/[SEP])
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    # 3. Forward pass -> per-token probabilities
    logits = model(**enc).logits[0]                 # (seq_len, num_labels)
    probs = torch.softmax(logits, dim=-1)
    pred_ids = probs.argmax(dim=-1).tolist()
    pred_scores = probs.max(dim=-1).values.tolist()

    # 4. One prediction per word: take the first sub-token (the one that had a label in training)
    tags = ["O"] * len(words)           # words cut off by truncation default to "O"
    scores = [1.0] * len(words)
    seen = set()
    for i, w in enumerate(word_ids):
        if w is None or w in seen:
            continue
        seen.add(w)
        tags[w] = id2label[pred_ids[i]]
        scores[w] = pred_scores[i]

    # 5. Merge BIO tags into entity spans
    entities, current = [], None
    for m, tag, score in zip(matches, tags, scores):
        if tag == "O":
            if current:
                entities.append(current); current = None
            continue
        prefix, label = tag.split("-", 1)
        if prefix == "B" or current is None or current["label"] != label:
            if current:
                entities.append(current)
            current = {"label": label, "start": m.start(), "end": m.end(), "scores": [score]}
        else:                            # "I-" tag continuing the same entity
            current["end"] = m.end()
            current["scores"].append(score)
    if current:
        entities.append(current)

    # 6. Final format: text slice from the original string, mean confidence
    return [
        {
            "text": text[e["start"]:e["end"]],
            "label": e["label"],
            "start": e["start"],
            "end": e["end"],
            "score": round(sum(e["scores"]) / len(e["scores"]), 4),
        }
        for e in entities
    ]

We can now use this method to identify all named entities in our example sentence. A well-trained model should find `John Smith` as `PER` (offsets 0-10), the first `Singapore` as `LOC` (20-29), and `National University of Singapore` as a single `ORG` (49-81), each with high confidence, while "2020" is most likely left untagged since CoNLL-2003 has no date type. The example shows why merging matters: multi-word entities come back as single spans rather than word-level tags, and the second "Singapore" is absorbed into the organization instead of being reported as a separate location. I can't run your model, so check the exact output against these expectations.

In [22]:
text = "John Smith lives in Singapore and works at the National University of Singapore since 2020."

for ent in predict_entities(text, model=model, tokenizer=tokenizer):
    print(f"{ent['label']:5s} {ent['score']:.3f}  {ent['text']!r} (start: {ent['start']}, end: {ent['end']})")

PER   0.999  'John Smith' (start: 0, end: 10)
LOC   0.999  'Singapore' (start: 20, end: 29)
ORG   0.996  'National University of Singapore' (start: 47, end: 79)


Instead of writing our own `predict_entities` function, the `transformers` library offers the `pipeline` abstraction, which bundles tokenization, the forward pass, and post-processing into a single ready-to-use object. This is the more practical route for inference, as it reduces the code we have to write and maintain, much like the `Trainer` does for training; as we will see later.

The code creates a pipeline for the `"token-classification"` task (the NER task) using our fine-tuned `model` and `tokenizer`, with `device` placing it on the same device as the model. The key argument is `aggregation_strategy="first"`, which tells the pipeline to do the merging for us: it groups subwords back into words, using the prediction of the first subword of each word (the same choice we made during training and in `predict_entities`), and merges consecutive BIO tags into complete entities. Without aggregation, the pipeline would return one prediction per subword token. Calling `ner(text)` then returns a list of dictionaries, one per entity, with keys such as `entity_group`, `score`, `word`, `start`, and `end`, which is essentially the same information as our manual implementation produces.

In [23]:
ner = pipeline("token-classification", 
               model=model, 
               tokenizer=tokenizer,
               aggregation_strategy="first",   # merges sub-words and BIO tags for you
               device=DEVICE)

Once the `pipeline` is created, you call it directly on the raw text, and it takes over everything predict_entities did:

In [24]:
for named_entity in ner(text):
    print(named_entity)

{'entity_group': 'PER', 'score': 0.9988072, 'word': 'John Smith', 'start': 0, 'end': 10}
{'entity_group': 'LOC', 'score': 0.9993437, 'word': 'Singapore', 'start': 20, 'end': 29}
{'entity_group': 'ORG', 'score': 0.9963272, 'word': 'National University of Singapore', 'start': 47, 'end': 79}


The validation metrics (precision, recall, and F1) and our short example sentence suggest that the fine-tuned model performs quite well on this kind of text. However, this is no guarantee that it will always perform well in practice. CoNLL-2003 consists of English Reuters news articles from a specific period, so the model has learned what entities look like in that style and domain. On very domain-specific content, such as biomedical, legal, or social media text, or on text with unusual formatting, entity types, or vocabulary, performance can drop considerably, and a handful of hand-picked examples cannot reveal this. If the target domain differs from the training data, it is therefore advisable to evaluate the model on representative in-domain data, and, if needed, to adapt it with additional domain-specific annotated data or a domain-specific pretrained model.

**Important:** Before training the second model with the `transformers` library, we release the memory still held by the first one. The custom model, its optimizer (AdamW keeps two extra state tensors per parameter), and the scheduler all keep tensors alive on the device, and if they stay allocated the second model may run out of memory or train more slowly. The cleanup works in three steps:

- **`del model, optimizer, scheduler`:** removes the Python references, since memory can only be freed once nothing points to it. Leftover variables such as `batch`, `loss`, or the logits should be deleted too.
- **`gc.collect()`:** runs Python's garbage collector to reclaim objects that are no longer referenced, including reference cycles.
- **`torch.cuda.empty_cache()` / `torch.mps.empty_cache()`:** PyTorch keeps freed GPU memory in a cache for reuse, so these calls return the unused blocks to the system. The first applies to NVIDIA GPUs and the second to Apple Silicon. Each is wrapped in `try/except`, so the cell runs on any machine, including a CPU-only one, where neither call is needed.

In [25]:
try:
    del model, optimizer, scheduler
except:
    pass
gc.collect()
try:
    torch.cuda.empty_cache()
except:
    pass
try:
    torch.mps.empty_cache()
except:
    pass

### Approach 2: `Trainer` Class

We now repeat the fine-tuning with the `Trainer` class, which takes over the training loop, optimization, scheduling, evaluation, and checkpointing, so we need far less code than in the manual approach. To make the comparison fair and start from scratch, we load a fresh copy of the pretrained model instead of reusing the one we already fine-tuned, which would otherwise continue training from its updated weights. The code is the same as before: `from_pretrained` loads the pretrained `bert-base-cased` encoder and attaches a new, randomly initialized token classification head with `num_labels=len(label_list)` outputs, `id2label` and `label2id` store our label mappings in the model configuration, and `.to(DEVICE)` moves the model to the chosen device. Reloading therefore resets the classification head to random values, so both approaches begin from the same starting point.

In [26]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

The `Trainer` takes care of the evaluation loop, but it does not know which metrics matter for NER. By default it only reports the loss, which says little about how well entities are actually recognized. The `compute_metrics()` function we pass to the `Trainer` fills this gap: after each evaluation, the `Trainer` calls it with the model's raw logits and the true label IDs for the whole validation set, and the function returns a dictionary of metrics that is logged and can be used to select the best checkpoint (e.g., via `metric_for_best_model="f1"`).

Inside, the function does the same conversion as our manual `evaluate()` method. It takes the `argmax` over the logits to get a predicted label ID per token, drops all positions labeled `-100` (special tokens, padding, and continuation subwords) so that one prediction per word remains, and converts the IDs into tag strings such as `B-PER` using `id2label`. Finally, it computes entity-level precision, recall, and F1 with `seqeval`, so the `Trainer` reports the same metrics as our manual training loop and the two approaches can be compared directly.

In [27]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    all_preds, all_refs = [], []
    for pred_row, label_row in zip(preds, labels):
        all_preds.append([id2label[p] for p, l in zip(pred_row, label_row) if l != -100])
        all_refs.append([id2label[l] for l in label_row if l != -100])

    return {
        "precision": precision_score(all_refs, all_preds),
        "recall": recall_score(all_refs, all_preds),
        "f1": f1_score(all_refs, all_preds)
    }

This code performs the actual fine-tuning with the `Trainer` class and uses the same main settings as the manual approach: a learning rate of `3e-5`, weight decay of `0.01`, a warmup of 10% of the training steps, and one epoch. This makes the two approaches directly comparable. The batch size of 32 and the logging every 50 steps also mirror the manual setup.

* **`TrainingArguments`** collects all the settings for the `Trainer`, using the same main values as the manual approach: a learning rate of 3e-5, weight decay of 0.01, a warmup of 10% of the steps (`warmup_steps=0.1`), and one epoch, together with a batch size of 32 for training and evaluation. `output_dir` is where checkpoints and logs are written, and `eval_strategy="epoch"` and `save_strategy="epoch"` evaluate on the validation set (using our `compute_metrics`) and save a checkpoint after every epoch. Since `load_best_model_at_end=False`, the final model is kept as is, so `metric_for_best_model="f1"` has no effect here and would only matter if the best checkpoint were restored at the end. Finally, `logging_steps=50` prints the training loss every 50 steps, and `report_to="none"` disables external trackers such as Weights & Biases. By default, the `Trainer` also uses AdamW with a linear warmup and decay schedule, so we do not need to build the optimizer and scheduler ourselves.

* **`Trainer`** ties everything together: the model, the arguments, the tokenized training and validation sets, the `data_collator` for dynamic padding, and `compute_metrics` for entity-level evaluation. It creates the `DataLoader`s, shuffles the training data, and handles the device placement, the forward and backward passes, gradient clipping (1.0 by default, as in our manual loop), and the optimizer and scheduler steps.

* **`trainer.train()`** starts the fine-tuning, which replaces our manual training loop with a single call. Finally, `torch.save(model.state_dict(), ...)` saves the fine-tuned weights to disk, as in the manual approach. Alternatively, `trainer.save_model(...)` would also save the configuration, including the label mappings, so the model can be reloaded with `from_pretrained`.

In [28]:
args = TrainingArguments(
    output_dir=output_folder,
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=0.1, 
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none"
)
 
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Start the training (i.e., fine-tuning)
trainer.train()

# Save model
torch.save(model.state_dict(), f"{output_folder}bert_ner.pt")

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.047491,0.042270,0.923732,0.931505,0.927602
2,0.031565,0.037794,0.938418,0.946314,0.942350
3,0.011689,0.033659,0.946461,0.952036,0.949241
4,0.008926,0.035510,0.943299,0.954729,0.948980
5,0.005551,0.036329,0.947219,0.954392,0.950792


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Having fine-tuned the model, we still need an unbiased estimate of how well it generalizes. The validation set was already used during training to monitor progress, and, had we used it for hyperparameter choices or checkpoint selection, it would no longer be completely independent. The test set, in contrast, has never influenced the model in any way, so it provides the final, honest measure of performance on unseen data. With the `Trainer`, this takes the following single call:

In [29]:
trainer.evaluate(tokenized_dataset["test"]);

Training Loss,Validation Loss,Epoch,Precision,Recall,F1
0.005551,0.118392,5,0.908092,0.921919,0.914953


With the `Trainer`-fine-tuned model, we can again wrap it in a `pipeline` to identify named entities in raw text, instead of writing our own post-processing. Creating a new pipeline is necessary here because it must point to the newly fine-tuned model, and applying it to the same example sentence also lets us check that the `Trainer` approach gives results comparable to the manual one. But otherwise the code is exactly the same:

In [30]:
ner = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="first",  # merges sub-words and BIO tags for you
)

And like before, we can test the pipeline using the same example sentence &mdash; and we should expect to see the exact same output.

In [31]:
text = "John Smith lives in Singapore and works at the National University of Singapore since 2020."

for named_entity in ner(text):
    print(named_entity)

{'entity_group': 'PER', 'score': 0.9986235, 'word': 'John Smith', 'start': 0, 'end': 10}
{'entity_group': 'LOC', 'score': 0.99960166, 'word': 'Singapore', 'start': 20, 'end': 29}
{'entity_group': 'ORG', 'score': 0.99423814, 'word': 'National University of Singapore', 'start': 47, 'end': 79}


---

## Summary

In this notebook, we fine-tuned a pretrained BERT model for NER and saw that a pretrained encoder with a small token classification head, trained on a modest dataset, can already recognize persons, organizations, locations, and miscellaneous entities well. More importantly, we learned why each step is necessary: why fine-tuning samples differ from pretraining samples, why subword tokenization requires label alignment with `-100`, why batches need padding, and why entity-level metrics such as precision, recall, and F1 are stricter than token-level accuracy. Comparing the manual loop with the `Trainer` made clear what happens under the hood and how much boilerplate the library removes.

Although we focused on NER, the setup is not specific to it and generalizes to other sequence labeling tasks, where every token (or word) in a text receives a label. Examples include part-of-speech tagging, syntactic chunking, slot filling in dialogue systems, and domain-specific tagging tasks such as extracting genes, drugs, or legal clauses. In each case, the recipe stays largely the same: load a pretrained encoder with `AutoModelForTokenClassification`, define the label set and the `label2id` and `id2label` mappings, tokenize into subwords and align the word-level labels, batch with dynamic padding, and train with a standard cross-entropy loss. Only the dataset, the number of labels, and the evaluation metric need to be adapted, for instance token-level accuracy for part-of-speech tagging instead of entity-level F1.

**Disclaimer:** This Jupyter Notebook contains content generated with the assistance of AI. While every effort has been made to review and validate the outputs, users should independently verify critical information before relying on it. The SELENE notebook repository is constantly evolving. We recommend downloading or pulling the latest version of this notebook from Github.